# 03. Predict gridded precipitation at 1, 2, 5, 10 km

Loads the final trained BayesNF VI model from
`s3://thesis-data-ismaktam/bayesnf/runs/vi__final__WY_h1_10__ffrk_full/`
and runs chunked predict over the monthly feature parquets produced by
notebooks 01 and 02.

For each resolution and each month a predictions parquet is written to
S3 with the per-cell mean and quantiles q05..q95.

`RESOLUTIONS_M` is configurable: set to a subset to predict only at the
resolutions of interest. Coarser grids run much faster (10 km is roughly
100x cheaper than 1 km). `SKIP_EXISTING=True` skips months already in S3
so a partial run can be resumed.


## 1. Environment and imports


In [ ]:
import os
# JAX env vars must be set before the first `import jax`
os.environ.setdefault('JAX_LOG_COMPILES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.90')

# Speedup flags (validated in speedup/speedup.ipynb): Triton GEMM, latency-hiding
# scheduler and async collectives. Smoke A/B matched tf32 loss to the 5th digit.
# os.environ['XLA_FLAGS'] = (
#     '--xla_gpu_enable_triton_gemm=true '
#     '--xla_gpu_enable_latency_hiding_scheduler=true '
#     '--xla_gpu_enable_async_collectives=true'
# )

import sys
import gc
import time
import json
import pickle
import threading
from pathlib import Path

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
import bayesnf
from bayesnf.spatiotemporal import (
    BayesianNeuralFieldVI,
    BayesianNeuralFieldMAP,
)

# bf16 mixed precision: enables tensor cores on A100/A40 at full bandwidth.
# Smoke A/B (notebooks/05_bayesnf/speedup/) kept ELBO finite and matched tf32
# loss to 5 significant digits, so it is numerically safe.
# Main benefit: ~1/2 peak activation memory, which keeps batch=131072 viable
# even on long (>10 year) training periods.
jax.config.update('jax_default_matmul_precision', 'bfloat16')
PRECISION_NOTE = 'bfloat16 (A100 tensor cores)'

# Repo root: notebooks/05_bayesnf/baseline/<this>.ipynb -> ../../..
ROOT = Path('../../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'cwd      : {os.getcwd()}')
print(f'bayesnf  : {getattr(bayesnf, "__version__", "n/a")}')
print(f'precision: {PRECISION_NOTE}')


## 2. GPU diagnostics


In [ ]:
import subprocess

backend = jax.default_backend()
devices = jax.devices()
print(f'JAX backend : {backend}')
print(f'JAX devices : {devices}')
print(f'n local     : {jax.local_device_count()}')

if backend != 'gpu':
    print('\nWARNING: JAX is on CPU - this notebook expects a GPU.')

try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    print('\nnvidia-smi:')
    print(smi.stdout)
except Exception as e:
    print(f'nvidia-smi unavailable: {e}')


## 3. Config


In [25]:
LIKELIHOOD    = 'ZINB'
TARGET_COL    = 'rainfall_int'
NEEDS_RESCALE = True
PRECIP_SCALE  = 10

RUN_NAME      = 'vi__final__WY_h1_10__ffrk_full'
RESOLUTIONS_M = [10000, 5000, 2000, 1000]   # coarsest first; drop entries to skip

# Predict only these calendar months in every year (1=Jan, 7=Jul). Full-year
# coverage is too expensive at 1/2 km — two representative months (one winter,
# one summer) are enough for the dataset chapter. Set to None to predict all 12.
MONTHS        = [3]

S3_BUCKET     = 'thesis-data-ismaktam'
S3_RUNS_ROOT  = 'bayesnf/runs'

LOCAL_RUNS = Path('results/bayesnf/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

# Per-resolution chunk size. Denser grids exhaust GPU memory inside
# `find_root_chandrupatla` (it materialises ensemble × chunk × mixture-comp
# scratch tensors). Empirical limit for ZINB on A100 (~40 GB):
#   10 km -> ~64 k cells/day  -> 500 k easily fits
#   5  km -> ~250 k cells/day -> 500 k fits
#   2  km -> ~1.6 M cells/day -> drop to 100 k
#   1  km -> ~6.5 M cells/day -> drop to 50 k
PRED_CHUNK_BY_RES = {
    1000:   50_000,
    2000:   50_000,
    5000:   50_000,
    10000:   50_000,
}
QUANTILES     = (0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95)
QUANTILE_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]

SKIP_EXISTING = True

print(f'run name      : {RUN_NAME}')
print(f'resolutions   : {RESOLUTIONS_M} m')
print(f'months        : {MONTHS if MONTHS is not None else "all"}')
print(f'output (s3)   : s3://{S3_BUCKET}/dataset/predictions_{{r}}km/YYYY-MM.parquet')


run name      : vi__final__WY_h1_10__ffrk_full
resolutions   : [10000, 5000, 2000, 1000] m
months        : [3]
output (s3)   : s3://thesis-data-ismaktam/dataset/predictions_{r}km/YYYY-MM.parquet


## 5. JIT-cache patch for `predict`

Same patch as in time_seasonality / geo_features. Without it `predict_bnf`
recompiles the closure on every chunk.


In [ ]:
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st
from tensorflow_probability.substrates import jax as _tfp_jax

_tfd = _tfp_jax.distributions
_LD  = bnf_models.LikelihoodDist

# --- bounded likelihood (anti-blowup for NB / ZINB) ---------------------------
# NB/ZINB parametrisation in bayesnf: mean = softplus(MLP(x)), shape = softplus(p1).
# MLP output is unbounded above, so on test rows with extreme features `mean`
# can blow up to ~10^6 mm. Guard: a thin tanh wrapper around the raw params.
#
#  raw_pred  -> PRED_CAP * tanh(raw_pred / PRED_CAP)   (soft clip; in the linear
#               regime when |raw_pred| << PRED_CAP it does not affect training).
#  params[1] -> SHAPE_CAP * tanh(params[1] / SHAPE_CAP) -> shape in ~[2.5e-3, 6]
#               -> total_count = 1/shape in ~[0.17, 400].
#
# Target range for mean in rainfall_int units: target <= 10*200 mm = 2000.
# PRED_CAP = 4000 (2x safety margin) -> softplus(+/-4000) ~ {~0, 4000}.
PRED_CAP  = 4000.0
SHAPE_CAP = 6.0

_orig_make_likelihood = bnf_models.make_likelihood_model


def make_likelihood_bounded(params, x, mlp, mlp_template, distribution):
    if _LD(distribution) == _LD.NORMAL:
        return _orig_make_likelihood(params, x, mlp, mlp_template, distribution)

    treedef    = jax.tree_util.tree_structure(mlp_template)
    mlp_params = jax.tree_util.tree_unflatten(treedef, params[3:])
    raw_pred   = mlp.apply(mlp_params, x)
    bounded_pred = PRED_CAP * jnp.tanh(raw_pred / PRED_CAP)
    mean       = jax.nn.softplus(bounded_pred)

    bounded_p1 = SHAPE_CAP * jnp.tanh(params[1] / SHAPE_CAP)
    shape      = jax.nn.softplus(bounded_p1)
    logits     = -jnp.log(shape) - jnp.log(mean)

    if _LD(distribution) == _LD.NB:
        return _tfd.Independent(
            _tfd.NegativeBinomial(total_count=1.0 / shape, logits=logits), 1)
    elif _LD(distribution) == _LD.ZINB:
        inflated_loc_probs = 1.0 / (1.0 + jnp.exp(-params[2]))
        return _tfd.Independent(
            _tfd.ZeroInflatedNegativeBinomial(
                total_count=1.0 / shape, logits=logits,
                inflated_loc_probs=inflated_loc_probs * jnp.ones(shape=mean.shape)), 1)
    raise AssertionError(f'unknown distribution: {distribution}')


bnf_models.make_likelihood_model = make_likelihood_bounded
print(f'bounded likelihood: mean <= ~{PRED_CAP:.0f} (rainfall_int units = '
      f'{PRED_CAP/PRECIP_SCALE:.0f} mm/day),  total_count in [{1/jax.nn.softplus(SHAPE_CAP):.3f}, '
      f'{1/jax.nn.softplus(-SHAPE_CAP):.1f}]')


# --- predict JIT-cache (same patch as in time_seasonality / geo_features) ----
def _patched_predict(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict
print('patched BayesianNeuralFieldEstimator.predict')


## 5b. Multi-GPU predict: larger inner batchsize

`bayesnf.inference.forecast_parameters_batched` processes the test data
in **batches of 1024 rows** in a Python loop. With PRED_CHUNK = 500_000
that is ~488 sequential pmap calls per chunk. On A100/A40 a forward
pass over 1024 rows is too small to saturate the cards — GPU utilisation
is low even though the cards look busy.

Solution: same pmap(vmap(vmap)) structure as the baseline, but with
`batchsize >= 16_384`. Lower Python overhead gives higher utilisation
across all cards and predict speeds up by 5-10x depending on the instance.

`N_DEVICES` is auto-detected via `jax.local_device_count()` so the same
notebook works on 1x / 4x / 8x GPU without changes.

Sanity: result is bitwise-equivalent to the baseline (same pmap structure,
only the inner batch size differs).


In [27]:
import jax
import jax.numpy as jnp
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st

N_DEVICES = jax.local_device_count()

# Per-device inner batch. Memory peak per device:
#   per_device_batch × ensemble_size × MLP_width × n_layers × dtype_bytes
# For ensemble_size=16, width=256, depth=2, float32 ≈ 130 KB/row → 16k rows ≈ 2GB
# Comfortable for 40-80GB cards; cap conservatively for smaller GPUs.
PER_DEVICE_BATCH = 16_384
PREDICT_BATCHSIZE = max(PER_DEVICE_BATCH * N_DEVICES, 16_384)

print(f'[predict-tune] N_DEVICES         = {N_DEVICES}')
print(f'[predict-tune] PER_DEVICE_BATCH  = {PER_DEVICE_BATCH:,}')
print(f'[predict-tune] PREDICT_BATCHSIZE = {PREDICT_BATCHSIZE:,}  '
      f'(was 1024 in baseline → ~{PREDICT_BATCHSIZE//1024}× larger)')


def _patched_predict_v2(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
        batchsize=PREDICT_BATCHSIZE,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict_v2
print(f'✓ predict re-patched with batchsize={PREDICT_BATCHSIZE:,} '
      f'across {N_DEVICES} device(s)')


[predict-tune] N_DEVICES         = 4
[predict-tune] PER_DEVICE_BATCH  = 16,384
[predict-tune] PREDICT_BATCHSIZE = 65,536  (was 1024 in baseline → ~64× larger)
✓ predict re-patched with batchsize=65,536 across 4 device(s)


## 5c. Per-cell upper bound for NB/ZINB quantile root-finder

Upstream `_get_nb_quantiles_root` (`bayesnf/inference.py:319-328`) uses

```
high = jnp.amax(dist.mean()) + 1.1/sqrt(1-q) * jnp.amax(dist.stddev())
```

`jnp.amax(...)` reduces over **every axis** → one scalar shared by all
cells in the chunk. If even one cell has a heavy-tailed NB (large
stddev), the upper bound becomes huge for everyone, Chandrupatla
bisection fails to converge in 60 iterations, and **every cell**
returns a value near that broken upper bound. Symptom on the map:
`q50 ≈ 500 mm` uniformly while `mean_mm ≈ 1–5 mm`.

Fix: switch to a per-cell upper bound by averaging `mean()` / `stddev()`
over the ensemble dimensions only, leaving the cell dimension intact.
One pathological cell no longer poisons the rest of the chunk.
Backward-compatible — model weights and ZINB/NB parameters are not
touched.


In [28]:
import bayesnf.inference as bnf_inference
from tensorflow_probability.substrates import jax as _tfp_jax


def _patched_nb_quantiles_root(dist, q, ensemble_axes=(0, 1, 2)):
    mean_c = dist.mean().mean(axis=ensemble_axes)
    std_c  = dist.stddev().mean(axis=ensemble_axes)
    high   = mean_c + 1.1 * jax.lax.rsqrt(1.0 - q) * std_c

    res = _tfp_jax.math.find_root_chandrupatla(
        lambda x: dist.cdf(x).mean(axis=ensemble_axes) - q,
        low=0.0,
        high=high,
        value_tolerance=1e-5,
        max_iterations=60,
    )
    return jnp.ceil(
        jnp.where(dist.prob(0).mean(axis=ensemble_axes) > q,
                  0, res.estimated_root)
    )


bnf_inference._get_nb_quantiles_root = _patched_nb_quantiles_root
print('✓ per-cell upper bound for NB/ZINB quantile root-finder')


✓ per-cell upper bound for NB/ZINB quantile root-finder


## 6. Load trained model from S3


In [29]:
import boto3
import pickle
from tensorflow_probability.python.internal import structural_tuple
import bayesnf.spatiotemporal as bnf_st

s3 = boto3.client('s3')

run_dir = LOCAL_RUNS / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
for fname in ('model.pkl', 'config.json'):
    p = run_dir / fname
    if not p.exists():
        s3.download_file(S3_BUCKET, f'{S3_RUNS_ROOT}/{RUN_NAME}/{fname}', str(p))

with open(run_dir / 'model.pkl', 'rb') as f:
    payload = pickle.load(f)

inference = payload['inference']
cls = bnf_st.BayesianNeuralFieldVI if inference == 'vi' else bnf_st.BayesianNeuralFieldMAP
model = cls(**payload['init_kwargs'])

StructT = structural_tuple.structtuple(payload['param_fields'])
model.params_ = StructT(*(payload['params_dict'][k] for k in payload['param_fields']))

cfg_run = json.loads((run_dir / 'config.json').read_text())
FEATURE_COLS = cfg_run['feature_cols']
YEAR_START   = cfg_run['year_start']
YEAR_END     = cfg_run['year_end']
print(f'  inference     : {inference}')
print(f'  feature_cols  : {FEATURE_COLS}')
print(f'  train period  : {YEAR_START}-{YEAR_END}')


  inference     : vi
  feature_cols  : ['datetime', 'latitude', 'longitude', 'elevation_m', 'idw', 'gos', 'svd_00', 'svd_01', 'svd_02', 'svd_03', 'svd_04', 'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa']
  train period  : 2014-2023


## 6b. Restore data_handler standardization stats

`model.pkl` does not persist `data_handler.mu_` / `std_` / `time_min_` /
`time_scale_` (the saved `standardize_` field is empty due to a bug in
`_save_run`). Without these, `predict` raises
`TypeError: unsupported operand type(s) for -: 'float' and 'NoneType'`
because `get_test` tries to do `(features - self.mu_) / self.std_`
with both ends `None`.

Fix: re-run `data_handler.get_train(df_train)` on the original training
data. This is the library's own routine for populating the four
internal arrays and guarantees the exact same standardization the model
was trained with.


In [30]:
DATA_DIR = Path('results/bayesnf/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_path = DATA_DIR / 'bayesnf_final_train.parquet'
if not train_path.exists():
    print(f'  downloading bayesnf_final_train.parquet ...')
    s3.download_file(S3_BUCKET, 'bayesnf/data/bayesnf_final_train.parquet',
                     str(train_path))

date_filter = [
    ('datetime', '>=', pd.Timestamp(f'{YEAR_START}-01-01')),
    ('datetime', '<=', pd.Timestamp(f'{YEAR_END}-12-31')),
]
keep = FEATURE_COLS + ['rainfall_int']
t0 = time.time()
df_train_for_stats = pd.read_parquet(train_path, filters=date_filter, columns=keep)
df_train_for_stats['datetime'] = pd.to_datetime(df_train_for_stats['datetime'])
print(f'  loaded {len(df_train_for_stats):,} rows in {time.time()-t0:.1f}s')

_ = model.data_handler.get_train(df_train_for_stats)
print(f'  data_handler populated:')
print(f'    mu_         shape={model.data_handler.mu_.shape}')
print(f'    std_        shape={model.data_handler.std_.shape}')
print(f'    time_min_   = {model.data_handler.time_min_}')
print(f'    time_scale_ = {model.data_handler.time_scale_}')

del df_train_for_stats
gc.collect()


  loaded 7,179,832 rows in 1.6s
  data_handler populated:
    mu_         shape=(17,)
    std_        shape=(17,)
    time_min_   = -2191
    time_scale_ = 3651.0


0

## 7. Predict per resolution


In [31]:
def _list_month_keys(prefix):
    keys = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=f'{prefix}/'):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.parquet'):
                keys.append(obj['Key'])
    keys = sorted(keys)
    if MONTHS is not None:
        # File names are 'YYYY-MM.parquet' → keep only those whose month
        # number is in the requested list.
        keys = [k for k in keys
                if int(Path(k).stem.split('-')[1]) in MONTHS]
    return keys


def _s3_key_exists(key):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except s3.exceptions.ClientError:
        return False


def _predict_df(df_feat, chunk_size):
    keep = FEATURE_COLS + ['rainfall_int']
    means_chunks, q_chunks = [], []
    n = len(df_feat)
    for i in range(0, n, chunk_size):
        j = min(i + chunk_size, n)
        m, q = model.predict(df_feat.iloc[i:j][keep],
                             quantiles=QUANTILES,
                             approximate_quantiles=True)
        means_chunks.append(np.asarray(m))
        q_chunks.append(np.asarray(q))

    means_raw = np.concatenate(means_chunks, axis=-1)
    q_raw     = np.concatenate(q_chunks,     axis=-1)
    mean_per_point = means_raw.reshape(-1, n).mean(axis=0)
    q_flat = q_raw.reshape(q_raw.shape[0], -1, n).mean(axis=1)

    if NEEDS_RESCALE:
        mean_per_point = mean_per_point / PRECIP_SCALE
        q_flat = q_flat / PRECIP_SCALE

    mean_per_point = np.clip(np.nan_to_num(mean_per_point, nan=0.0,
                                            posinf=250.0, neginf=0.0),
                              0.0, 250.0)
    q_flat = np.clip(np.nan_to_num(q_flat, nan=0.0, posinf=500.0, neginf=0.0),
                     0.0, 500.0)
    return mean_per_point, q_flat


for res_m in RESOLUTIONS_M:
    res_km = res_m // 1000
    src_prefix = f'dataset/grid_{res_km}km'
    out_prefix = f'dataset/predictions_{res_km}km'
    pred_chunk = PRED_CHUNK_BY_RES[res_m]

    keys = _list_month_keys(src_prefix)
    print(f'\n=== {res_km} km: {len(keys)} months  (chunk={pred_chunk:,}) ===')

    local_src = Path(f'results/dataset/grid_{res_km}km')
    local_out = Path(f'results/dataset/predictions_{res_km}km')
    local_src.mkdir(parents=True, exist_ok=True)
    local_out.mkdir(parents=True, exist_ok=True)

    for src_key in keys:
        month_name = Path(src_key).name
        out_key = f'{out_prefix}/{month_name}'

        if SKIP_EXISTING and _s3_key_exists(out_key):
            continue

        local_feat = local_src / month_name
        if not local_feat.exists():
            s3.download_file(S3_BUCKET, src_key, str(local_feat))
        df_feat = pd.read_parquet(local_feat)

        t0 = time.time()
        mean_mm, q_flat = _predict_df(df_feat, pred_chunk)
        elapsed = time.time() - t0

        out_df = df_feat[['datetime', 'cell_id', 'latitude', 'longitude',
                          'x_proj', 'y_proj']].copy()
        out_df['mean_mm'] = mean_mm
        for lbl, qv in zip(QUANTILE_LABELS, q_flat):
            out_df[f'q{lbl:02d}'] = qv

        out_local = local_out / month_name
        out_df.to_parquet(out_local, index=False)
        s3.upload_file(str(out_local), S3_BUCKET, out_key)
        print(f'  {res_km} km / {month_name}: {len(out_df):>10,} rows  {elapsed:.0f}s')

        del df_feat, out_df, mean_mm, q_flat
        gc.collect()
        jax.clear_caches()



=== 10 km: 1 months  (chunk=50,000) ===
  10 km / 2023-03.parquet:     64,108 rows  61s

=== 5 km: 1 months  (chunk=50,000) ===
  5 km / 2023-03.parquet:    253,704 rows  334s

=== 2 km: 1 months  (chunk=50,000) ===
  2 km / 2023-03.parquet:  1,575,420 rows  1432s

=== 1 km: 1 months  (chunk=50,000) ===
  1 km / 2023-03.parquet:  6,287,358 rows  5136s
